# 01 — DBpedia query collection

**Purpose:** Fetches SPARQL queries from the LSQ endpoint (DBpedia-backed), runs them against the DBpedia endpoint, and keeps only those that return results.

**Output:** `../successful_queries/successful_queries_dbpedia.json` (list of `{"sparql_query": "..."}`).

**Run before:** `02_dbpedia_semantic_triplets.ipynb`.

### 1. Imports

In [1]:
import json
import time
import pandas as pd
import requests
from tqdm import tqdm
from SPARQLWrapper import SPARQLWrapper, JSON

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/ganesh-jupyterhub/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2. LSQ endpoint and query fetch

In [2]:
LSQ_ENDPOINT = "http://lsq.aksw.org/sparql"
QUERY_TEMPLATE = """
PREFIX lsqv: <http://lsq.aksw.org/vocab#> 
PREFIX sd: <http://www.w3.org/ns/sparql-service-description#> 

SELECT DISTINCT ?text
WHERE {{ 
   ?query lsqv:hasRemoteExec/lsqv:endpoint <https://dbpedia.org/sparql> .
   ?query lsqv:text ?text . 
   ?query lsqv:hasSpin ?spin .
   ?spin a <http://spinrdf.org/sp#Select> . 
   ?query lsqv:hasStructuralFeatures ?features .

   FILTER NOT EXISTS {{ ?features lsqv:usesFeature lsqv:Count . }}

   FILTER(!CONTAINS(LCASE(STR(?text)), "dbpo:wikipageid"))
}}
LIMIT 1000
OFFSET {offset}
"""

def run_lsq_query(offset):
    sparql = SPARQLWrapper(LSQ_ENDPOINT)
    sparql.setQuery(QUERY_TEMPLATE.format(offset=offset))
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    return [result["text"]["value"] for result in results["results"]["bindings"]]

def fetch_all_queries(num_batches=60, batch_size=1000, delay=0.5):
    all_results = []
    for i in range(num_batches):
        offset = i * batch_size
        print(f"Fetching batch {i+1} with OFFSET {offset}...")
        try:
            rows = run_lsq_query(offset)
            all_results.extend(rows)
        except Exception as e:
            print(f"Error at offset {offset}: {e}")
        time.sleep(delay)
    return all_results

### 3. Fetch, save to CSV, and load

In [3]:
queries = fetch_all_queries()
pd.DataFrame(queries, columns=["text"]).to_csv("sparql_results.csv", index=False, encoding="utf-8")
print("Results saved to sparql_results.csv")

df = pd.read_csv("sparql_results.csv")
print(f"Total queries loaded: {len(df)}")

Fetching batch 1 with OFFSET 0...
Fetching batch 2 with OFFSET 1000...
Fetching batch 3 with OFFSET 2000...
Fetching batch 4 with OFFSET 3000...
Fetching batch 5 with OFFSET 4000...
Fetching batch 6 with OFFSET 5000...
Fetching batch 7 with OFFSET 6000...
Fetching batch 8 with OFFSET 7000...
Fetching batch 9 with OFFSET 8000...
Fetching batch 10 with OFFSET 9000...
Fetching batch 11 with OFFSET 10000...
Fetching batch 12 with OFFSET 11000...
Fetching batch 13 with OFFSET 12000...
Fetching batch 14 with OFFSET 13000...
Fetching batch 15 with OFFSET 14000...
Fetching batch 16 with OFFSET 15000...
Fetching batch 17 with OFFSET 16000...
Fetching batch 18 with OFFSET 17000...
Fetching batch 19 with OFFSET 18000...
Fetching batch 20 with OFFSET 19000...
Fetching batch 21 with OFFSET 20000...
Fetching batch 22 with OFFSET 21000...
Fetching batch 23 with OFFSET 22000...
Fetching batch 24 with OFFSET 23000...
Fetching batch 25 with OFFSET 24000...
Fetching batch 26 with OFFSET 25000...
Fetching

### 4. Duplicate analysis

In [4]:
total_rows = len(df)
unique_rows = df["text"].nunique()
duplicate_count = total_rows - unique_rows

print(f"Total rows: {total_rows}")
print(f"Unique rows: {unique_rows}")
print(f"Duplicate rows: {duplicate_count}")

duplicates = df[df.duplicated("text", keep=False)].sort_values("text")
print("\nSample duplicates:")
print(duplicates.head(10))

Total rows: 59000
Unique rows: 59000
Duplicate rows: 0

Sample duplicates:
Empty DataFrame
Columns: [text]
Index: []


### 5. DBpedia endpoint helper

In [5]:
def query_dbpedia(sparql_query):
    endpoint = "https://dbpedia.data.dice-research.org/sparql"
    headers = {"Accept": "application/sparql-results+json"}
    params = {"query": sparql_query}
    try:
        response = requests.get(endpoint, headers=headers, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException:
        return None

### 6. Filter successful queries and save

In [6]:
successful_queries = []
failed = 0

for index, row in tqdm(df.iterrows(), total=len(df), desc="Querying DBpedia"):
    sparql_query = row.get("text")
    if sparql_query:
        results = query_dbpedia(sparql_query)
        if results and "results" in results and results["results"]["bindings"]:
            successful_queries.append({"sparql_query": sparql_query})
        else:
            failed += 1

print(f"Successful queries: {len(successful_queries)}")
print(f"Failed queries: {failed}")

with open("../successful_queries/successful_queries_dbpedia.json", "w", encoding="utf-8") as f:
    json.dump(successful_queries, f, ensure_ascii=False, indent=2)

Querying DBpedia: 100%|██████████| 59000/59000 [1:17:32<00:00, 12.68it/s]   


Successful queries: 16450
Failed queries: 42550
